# Carnet 3 : L'Automatisation du Tuning avec MLflow

Dans le carnet précédent, nous avons vu que tester des hyperparamètres à la main ou avec une boucle aléatoire naïve était fastidieux, illisible dans MLflow et surtout sous-optimal.

L'objectif de ce carnet est de laisser l'ordinateur trouver la configuration parfaite lui-même. Nous allons comparer trois bibliothèques très connues pour optimiser notre modèle XGBoost :
- **GridSearchCV** : L'approche classique (exhaustive mais lente).
- **Hyperopt** : Un framework d'optimisation bayésienne populaire (TPE).
- **Optuna** : Un framework moderne, intelligent (optimisation bayésienne), extrêmement rapide et qui s'intègre parfaitement avec MLflow.

Pour rappel, notre critère métier principal est le **Recall** (pour rater le moins d'accidents graves possible).

## 1. Imports et Chargement des Données

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from datetime import datetime
import tempfile
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, log_loss, ConfusionMatrixDisplay
import xgboost as xgb
from xgboost import XGBClassifier

import mlflow
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll.base import scope

import optuna
from optuna.integration.mlflow import MLflowCallback

# Configuration de base MLflow
mlflow.set_tracking_uri("http://localhost:5000")

# Chargement et préparation habituelle
df_accident = pd.read_csv('data/dataset_accident.csv', sep=';' )
y = df_accident["grav_binary"]
X = df_accident.drop(columns=["grav_ordered", "grav_binary"])

# Nouveauté : On ajoute 'stratify=y' pour garantir la même proportion d'accidents graves 
# dans le train et le test. Très utile pour l'optimisation avancée !
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

## 2. Approche 1 : GridSearchCV

C'est la méthode "force brute". On lui donne une grille de valeurs, et elle va tester absolument toutes les combinaisons possibles. 

L'avantage de `mlflow.sklearn.autolog()` est qu'il tracke tout automatiquement pendant la recherche de grille !

In [ ]:
experiment_name="AutoTuning"
mlflow.set_experiment(experiment_name)

# Activation de l'autologging pour GridSearchCV (sans le modèle, qu'on logge manuellement)
mlflow.sklearn.autolog(log_models=False)

param_grid = {
    'n_estimators': [100, 200, 500],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 9],
    'subsample': [0.7, 1.0]
}

# Scoring multi-métriques pour avoir les 5 scores par combinaison en CV
scoring = {
    'recall': 'recall_weighted',
    'accuracy': 'accuracy',
    'f1_score': 'f1_weighted',
    'precision': 'precision_weighted',
    'log_loss': 'neg_log_loss'
}

print("Grille définie : 3 x 3 x 3 x 2 = 54 combinaisons à tester.")
print("Avec une validation croisée de 3 (cv=3), cela fait 162 entraînements de modèles. Soyez patient...")

with mlflow.start_run(run_name=f"GridSearch {datetime.now().strftime('%d/%m/%Y %Hh:%Mm:%Ss')}"):
    xgb_model = XGBClassifier(eval_metric="logloss")
    
    grid_search = GridSearchCV(
        estimator=xgb_model,
        param_grid=param_grid,
        cv=3,
        scoring=scoring,
        refit='recall',
        verbose=1,
        n_jobs=1
    )

    grid_search.fit(X_train, y_train)
    
    # Log des nested runs pour chaque combinaison avec les 5 métriques CV
    results = grid_search.cv_results_
    for i in range(len(results['params'])):
        with mlflow.start_run(nested=True, run_name=f"grid_{i}"):
            mlflow.log_params(results['params'][i])
            mlflow.log_metrics({
                "accuracy": results['mean_test_accuracy'][i],
                "f1_score": results['mean_test_f1_score'][i],
                "precision": results['mean_test_precision'][i],
                "recall": results['mean_test_recall'][i],
                "log_loss": -results['mean_test_log_loss'][i]
            })
    
    # Métriques best_* sur le train
    y_pred_train = grid_search.best_estimator_.predict(X_train)
    y_pred_proba_train = grid_search.best_estimator_.predict_proba(X_train)
    
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metrics({
        "train_accuracy": accuracy_score(y_train, y_pred_train),
        "train_f1_score": f1_score(y_train, y_pred_train, average='weighted'),
        "train_precision": precision_score(y_train, y_pred_train, average='weighted'),
        "train_recall": recall_score(y_train, y_pred_train, average='weighted'),
        "train_log_loss": log_loss(y_train, y_pred_proba_train)
    })
    
    # Métriques test_* sur le test set
    y_pred = grid_search.best_estimator_.predict(X_test)
    y_pred_proba = grid_search.best_estimator_.predict_proba(X_test)
    
    mlflow.log_metrics({
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_f1_score": f1_score(y_test, y_pred, average='weighted'),
        "test_precision": precision_score(y_test, y_pred, average='weighted'),
        "test_recall": recall_score(y_test, y_pred, average='weighted'),
        "test_log_loss": log_loss(y_test, y_pred_proba)
    })
    
    # Sauvegarde du modèle sous un nom distinctif
    mlflow.xgboost.log_model(grid_search.best_estimator_, name="XGBoost_GridSearch_Best")

    # Matrice de confusion du meilleur modèle
    with tempfile.TemporaryDirectory() as tmpdir:
        fig, ax = plt.subplots(figsize=(8, 6))
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
        ax.set_title("Confusion Matrix - GridSearch Best")
        plt.tight_layout()
        cm_path = os.path.join(tmpdir, "confusion_matrix.png")
        plt.savefig(cm_path)
        mlflow.log_artifact(cm_path)
        plt.close()
 
    print("\n--- Résultats GridSearchCV ---")
    print(f"Meilleur recall en CV : {grid_search.best_score_:.4f}")
    print(f"Meilleurs paramètres  : {grid_search.best_params_}")

## 3. Approche 2 : Hyperopt

Hyperopt est une bibliothèque d'optimisation bayésienne très populaire, plus ancienne qu'Optuna. Elle utilise l'algorithme TPE (Tree-structured Parzen Estimator) pour minimiser une fonction de perte. Dans notre cas, nous voulons maximiser le Recall, donc nous minimisons son opposé (`-score`).

Contrairement à Optuna, la journalisation des paramètres et métriques avec MLflow doit se faire **manuellement** à travers la définition de la fonction à minimiser.

In [ ]:
mlflow.set_experiment(experiment_name)

# On désactive l'autolog pour garder le contrôle
mlflow.sklearn.autolog(disable=True) 

def objective_hyperopt(params):
    # Chaque évaluation de paramètres devient un "nested run" (enfant)
    with mlflow.start_run(nested=True):
        model = xgb.XGBClassifier(**params, random_state=42, eval_metric="logloss")
        model.fit(X_train, y_train)
        
        preds = model.predict(X_test)
        preds_proba = model.predict_proba(X_test)
        
        rec = recall_score(y_test, preds, average='weighted')
        
        # Log manuel
        mlflow.log_params(params)
        mlflow.log_metrics({
            "accuracy": accuracy_score(y_test, preds),
            "f1_score": f1_score(y_test, preds, average='weighted'),
            "precision": precision_score(y_test, preds, average='weighted'),
            "recall": rec,
            "log_loss": log_loss(y_test, preds_proba)
        })
        
        # Hyperopt cherche toujours à *minimiser* !
        return {'loss': -rec, 'status': STATUS_OK}

# Définition de l'espace de recherche (qui doit être spécifié à l'avance, contrairement à Optuna)
space = {
    'n_estimators': scope.int(hp.quniform('n_estimators', 100, 1000, 1)),
    'max_depth': scope.int(hp.quniform('max_depth', 3, 12, 1)),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
    'subsample': hp.uniform('subsample', 0.5, 1.0)
}

with mlflow.start_run(run_name=f"Hyperopt {datetime.now().strftime('%d/%m/%Y %Hh:%Mm:%Ss')}"):
    trials = Trials()
    print("Lancement d'Hyperopt. Il va effectuer 20 expériences.")
    best = fmin(fn=objective_hyperopt, space=space, algo=tpe.suggest, max_evals=20, trials=trials)
    
    # hp.quniform peut renvoyer des float, donc on caste pour XGBoost
    best_params = {
        'n_estimators': int(best['n_estimators']),
        'max_depth': int(best['max_depth']),
        'learning_rate': best['learning_rate'],
        'subsample': best['subsample']
    }
    
    # On log les infos du master run
    mlflow.log_params(best_params)
    
    # Entraînement final et sauvegarde du meilleur modèle
    best_model = xgb.XGBClassifier(**best_params, random_state=42, eval_metric="logloss")
    best_model.fit(X_train, y_train)
    
    # Métriques best_* sur le train
    y_pred_train = best_model.predict(X_train)
    y_pred_proba_train = best_model.predict_proba(X_train)
    
    mlflow.log_metrics({
        "training_accuracy": accuracy_score(y_train, y_pred_train),
        "training_f1_score": f1_score(y_train, y_pred_train, average='weighted'),
        "training_precision": precision_score(y_train, y_pred_train, average='weighted'),
        "training_recall": recall_score(y_train, y_pred_train, average='weighted'),
        "training_log_loss": log_loss(y_train, y_pred_proba_train)
    })
    
    # Métriques test_* sur le test set
    y_pred = best_model.predict(X_test)
    y_pred_proba = best_model.predict_proba(X_test)
    
    mlflow.log_metrics({
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_f1_score": f1_score(y_test, y_pred, average='weighted'),
        "test_precision": precision_score(y_test, y_pred, average='weighted'),
        "test_recall": recall_score(y_test, y_pred, average='weighted'),
        "test_log_loss": log_loss(y_test, y_pred_proba)
    })
    
    mlflow.xgboost.log_model(best_model, name="XGBoost_Hyperopt_Best")

    # Matrice de confusion du meilleur modèle
    with tempfile.TemporaryDirectory() as tmpdir:
        fig, ax = plt.subplots(figsize=(8, 6))
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
        ax.set_title("Confusion Matrix - Hyperopt Best")
        plt.tight_layout()
        cm_path = os.path.join(tmpdir, "confusion_matrix.png")
        plt.savefig(cm_path)
        mlflow.log_artifact(cm_path)
        plt.close()
    
    print("\n--- Résultats Hyperopt ---")
    print(f"✅ Optimisation terminée. Meilleur Recall (test) : {mlflow.get_run(mlflow.active_run().info.run_id).data.metrics['test_recall']:.4f}")

## 4. Approche 3 : Optuna

GridSearch est long... Optuna est intelligent ! Au lieu de quadriller, Optuna va essayer une configuration, voir le résultat, et ajuster mathématiquement sa prochaine supposition (Optimisation Bayésienne).

De plus, nous pouvons utiliser `MLflowCallback` très simplement pour que chaque itération (ou *trial*) d'Optuna atterrisse joliment dans notre dashboard MLflow en tant qu'enfant (nested run).

In [ ]:
# Désactivation de l'autolog pour garder un MLflow propre avec Optuna
mlflow.sklearn.autolog(disable=True) 

mlflow.set_experiment(experiment_name)

def objective(trial):
    # Optuna 'suggest' les paramètres de façon large et continue, sans grille fixe
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'eval_metric': 'logloss'
    }
    
    model = xgb.XGBClassifier(**params, random_state=42)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    preds_proba = model.predict_proba(X_test)
    
    rec = recall_score(y_test, preds, average='weighted')
    
    # Log manuel dans un nested run (comme Hyperopt, plus fiable que le callback)
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):
        mlflow.log_params(params)
        mlflow.log_metrics({
            "accuracy": accuracy_score(y_test, preds),
            "f1_score": f1_score(y_test, preds, average='weighted'),
            "precision": precision_score(y_test, preds, average='weighted'),
            "recall": rec,
            "log_loss": log_loss(y_test, preds_proba)
        })
    
    return rec


with mlflow.start_run(run_name=f"Optuna {datetime.now().strftime('%d/%m/%Y %Hh:%Mm:%Ss')}"):

    # direction="maximize" car on veut augmenter le Recall (direction="minimize" pour Log Loss par exemple)
    study = optuna.create_study(direction="maximize")
    print("Lancement d'Optuna. Il va faire 20 essais avec une intelligence bayésienne.")
    study.optimize(objective, n_trials=20)

    # --- Fin de l'étude, on logge le champion ! ---
    mlflow.log_params(study.best_params)
    
    # Ré-entraînement sur les meilleurs hyperparamètres trouvés
    best_model = xgb.XGBClassifier(**study.best_params, random_state=42, eval_metric="logloss")
    best_model.fit(X_train, y_train)
    
    # Métriques best_* sur le train
    y_pred_train = best_model.predict(X_train)
    y_pred_proba_train = best_model.predict_proba(X_train)
    
    mlflow.log_metrics({
        "training_accuracy": accuracy_score(y_train, y_pred_train),
        "training_f1_score": f1_score(y_train, y_pred_train, average='weighted'),
        "training_precision": precision_score(y_train, y_pred_train, average='weighted'),
        "training_recall": recall_score(y_train, y_pred_train, average='weighted'),
        "training_log_loss": log_loss(y_train, y_pred_proba_train)
    })
    
    # Métriques test_* sur le test set
    y_pred = best_model.predict(X_test)
    y_pred_proba = best_model.predict_proba(X_test)
    
    mlflow.log_metrics({
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_f1_score": f1_score(y_test, y_pred, average='weighted'),
        "test_precision": precision_score(y_test, y_pred, average='weighted'),
        "test_recall": recall_score(y_test, y_pred, average='weighted'),
        "test_log_loss": log_loss(y_test, y_pred_proba)
    })
    
    # Sauvegarde de ce champion
    mlflow.xgboost.log_model(best_model, name="XGBoost_Optuna_Best")

    # Matrice de confusion du meilleur modèle
    with tempfile.TemporaryDirectory() as tmpdir:
        fig, ax = plt.subplots(figsize=(8, 6))
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
        ax.set_title("Confusion Matrix - Optuna Best")
        plt.tight_layout()
        cm_path = os.path.join(tmpdir, "confusion_matrix.png")
        plt.savefig(cm_path)
        mlflow.log_artifact(cm_path)
        plt.close()
    
    print("\n--- Résultats Optuna ---")
    print(f"✅ Optimisation terminée. Meilleur Recall (test) : {study.best_value:.4f}")
    print("Modèle 'XGBoost_Optuna_Best' enregistré dans MLflow.")

## 5. Bilan du Comparatif

Ouvrez le **Dashboard MLflow** (http://127.0.0.1:5000).

Vous pouvez comparer les résultats des trois méthodes. Observez comment chaque librairie crée ses plans de tests et avec quelle efficacité elle trouve ou non le meilleur Recall.

### Tableau Comparatif

Voici un tableau comparatif à compléter selon vos observations :

| Méthode | Avantages | Inconvénients | Score Recall (Meilleur) |
| :--- | :--- | :--- | :--- |
| **GridSearchCV** | Exhaustif, facile à comprendre, bien géré par `autolog()`. | Très couteux en calcul. Patauge si l'espace est grand. Risque de crash avec multiprocessing et XGBoost. | *À compléter* |
| **Hyperopt** | Optimisation mathématique intelligente (Bayésienne). Rapide. | Syntaxe complexe (définition de l'`espace`). L'intégration MLflow est manuelle et parfois laborieuse. | *À compléter* |
| **Optuna** | Intelligence Bayésienne, très performant. Syntaxe dynamique élégante. Callback MLflow `MLflowCallback` parfait. | Nécessite d'installer `optuna` et `optuna-integration`. | *À compléter* |

### Conclusion
**Optuna** s'impose aujourd'hui comme le standard de l'industrie pour sa simplicité, la beauté de son code et sa redoutable efficacité, reléguant peu à peu Hyperopt ou GridSearchCV aux oubliettes pour le *Tuning* agressif.